# iSCORS-Net — Fast Runner

**Workflow:**
1. Run **Setup** — clones the repo and installs deps.
2. Run **Train** — internal learning on the test video.
3. Run **Results** — inline visualisation.
4. Run **Download** — saves `results_<VERSION>.zip` to your machine.

---

In [ ]:
VERSION = 'v3.0'
print(f'iSCORS-Net {VERSION}')

In [ ]:
import os

REPO   = 'https://github.com/breezy90126/iscors-net.git'
BRANCH = 'claude/beautiful-volta-gFUot'

if not os.path.isdir('iscors-net'):
    !git clone --depth 1 -b {BRANCH} {REPO}
else:
    !git -C iscors-net pull

os.chdir('iscors-net')
print('Working dir:', os.getcwd())

!pip install -q -r requirements.txt scipy
print('Dependencies installed.')

In [ ]:
import os

# Always regenerate — ensures new background design (near-static) is used
video_path = './data/test_synthetic_cell.tif'
if os.path.exists(video_path):
    os.remove(video_path)
    print('Removed old video.')

os.makedirs('./data', exist_ok=True)
!python utils/generate_test_video.py

In [ ]:
!python train_phys_recon.py

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob, os

result_files = sorted(glob.glob('./result/*.png'))
print(f'Result images ({len(result_files)}):', [os.path.basename(f) for f in result_files])

for path in result_files:
    img = mpimg.imread(path)
    plt.figure(figsize=(10, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.title(os.path.basename(path))
    plt.tight_layout()
    plt.show()

In [ ]:
import zipfile, os, glob
from google.colab import files

zip_name = f'results_{VERSION}.zip'

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for pattern in [f'./result/*{VERSION}*', f'./checkpoint/*{VERSION}*']:
        for path in glob.glob(pattern):
            zf.write(path, os.path.relpath(path, '.'))
    for path in glob.glob('./result/*.png'):
        arcname = os.path.relpath(path, '.')
        if arcname not in zf.namelist():
            zf.write(path, arcname)

print(f'Created {zip_name}  ({os.path.getsize(zip_name)/1024:.1f} KB)')
files.download(zip_name)

---

## Real Cell Analysis

**Workflow:**
1. **real-setup** — Mount Google Drive, unzip `large_file.zip`, locate files.
2. **real-mat** — Decode `Output_iSCORS_map.mat` → TIF reference maps (not fed to model).
3. **real-preprocess** — Normalise video: ÷ temporal median → ÷ per-frame Gaussian (σ=4).
4. **real-inference** — Run trained model on preprocessed video.
5. **real-compare** — Side-by-side: model predictions vs traditional iSCORS reference.


In [ ]:
# ── Real Cell Analysis: Mount Drive & Extract ─────────────────────────────
from google.colab import drive
import zipfile, os

drive.mount('/content/drive', force_remount=False)

ZIP_PATH    = '/content/drive/MyDrive/iscors_test/large_file.zip'
EXTRACT_DIR = '/content/real_data'
SAVE_DIR    = '/content/drive/MyDrive/iscors_test'

os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(SAVE_DIR,    exist_ok=True)

print(f'Extracting {ZIP_PATH} ...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_DIR)
print('Extraction done.')

def find_file(root, name):
    for dirpath, _, files in os.walk(root):
        if name in files:
            return os.path.join(dirpath, name)
    return None

VIDEO_PATH = find_file(EXTRACT_DIR, 'COBRI_rarw_video.tif')
MAT_PATH   = find_file(EXTRACT_DIR, 'Output_iSCORS_map.mat')

print(f'Video : {VIDEO_PATH}')
print(f'MAT   : {MAT_PATH}')
assert VIDEO_PATH, 'COBRI_rarw_video.tif not found in zip!'
assert MAT_PATH,   'Output_iSCORS_map.mat not found in zip!'


In [ ]:
# ── Decode Output_iSCORS_map.mat → TIF (reference only, not fed to model) ──
import numpy as np
import tifffile

# Try scipy.io (MATLAB v5/v7); fall back to h5py (v7.3 / HDF5)
gamma_ref = alpha_ref = None
try:
    import scipy.io as sio
    mat = sio.loadmat(MAT_PATH)
    data = {k: v for k, v in mat.items() if not k.startswith('_')}
    print('Loaded via scipy.io.  Fields:', list(data.keys()))
    _h5 = False
except Exception as e:
    print(f'scipy.io failed ({e}), trying h5py ...')
    import h5py
    _h5_file = h5py.File(MAT_PATH, 'r')
    data = {k: _h5_file[k] for k in _h5_file}
    print('Loaded via h5py.  Fields:', list(data.keys()))
    _h5 = True

def _find(d, *names):
    lmap = {k.lower(): k for k in d}
    for n in names:
        if n.lower() in lmap:
            return np.array(d[lmap[n.lower()]]).squeeze().astype(np.float32)
    return None

gamma_ref = _find(data, 'gamma', 'Gamma', 'gamma_map', 'D')
alpha_ref = _find(data, 'alpha', 'Alpha', 'alpha_map', 'beta', 'anomalous_exp')

if _h5:
    _h5_file.close()

for name, arr in [('gamma_ref', gamma_ref), ('alpha_ref', alpha_ref)]:
    if arr is not None:
        print(f'{name}: shape={arr.shape}  range=[{arr.min():.4f}, {arr.max():.4f}]')
    else:
        print(f'{name}: field not auto-detected — inspect data keys above and set manually')

# Save reference TIFs to Drive
if gamma_ref is not None:
    p = os.path.join(SAVE_DIR, 'iscors_gamma_ref.tif')
    tifffile.imwrite(p, gamma_ref)
    print(f'Saved -> {p}')
if alpha_ref is not None:
    p = os.path.join(SAVE_DIR, 'iscors_alpha_ref.tif')
    tifffile.imwrite(p, alpha_ref)
    print(f'Saved -> {p}')


In [ ]:
# ── Preprocess Real Video ──────────────────────────────────────────────────
# 1. Chunked 2×2 spatial binning at read time  (memory-efficient)
# 2. Flat-field  : divide by per-pixel temporal median
# 3. BG removal  : divide each frame by Gaussian-smoothed self (sigma=4)
#
# Signal centering note
#   compute_g_empirical_map divides by mean_I².
#   video_proc keeps mean ≈ 1.0  → G formula is valid.
#   (video_proc − 1) is ONLY for visualisation of zero-centred fluctuations.
import numpy as np
import tifffile
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

CHUNK_SIZE = 100
N_FRAMES   = 2000

# ── Quick preview of raw frame 0 (before binning) ───────────────────────────
frame0_raw = tifffile.imread(VIDEO_PATH, key=0).astype(np.float32)
H_orig, W_orig = frame0_raw.shape
H_bin, W_bin   = H_orig // 2, W_orig // 2
print(f'Original frame shape : {H_orig} × {W_orig}')
print(f'After 2×2 binning    : {H_bin}  × {W_bin}')

fig, ax = plt.subplots(figsize=(6, 5))
p1, p99 = np.percentile(frame0_raw, 1), np.percentile(frame0_raw, 99)
im = ax.imshow(frame0_raw, cmap='gray', vmin=p1, vmax=p99)
plt.colorbar(im, ax=ax, label='Intensity')
ax.set_title(f'Frame 0 — raw  ({H_orig}×{W_orig})')
ax.axis('off')
plt.tight_layout()
plt.show()

# ── Chunked read + 2×2 binning ──────────────────────────────────────────────
with tifffile.TiffFile(VIDEO_PATH) as tif:
    total_frames = len(tif.pages)

n_frames  = min(N_FRAMES, total_frames)
video_raw = np.empty((n_frames, H_bin, W_bin), dtype=np.float32)

print(f'\nReading {n_frames} frames in chunks of {CHUNK_SIZE} ...')
for start in range(0, n_frames, CHUNK_SIZE):
    end   = min(start + CHUNK_SIZE, n_frames)
    chunk = tifffile.imread(VIDEO_PATH, key=range(start, end)).astype(np.float32)
    T_c   = end - start
    video_raw[start:end] = chunk.reshape(T_c, H_bin, 2, W_bin, 2).mean(axis=(2, 4))
    del chunk
    if start % (CHUNK_SIZE * 4) == 0:
        print(f'  frames {start:4d}–{end-1:4d}  /  {n_frames}')

T, H, W = video_raw.shape
print(f'\nLoaded (binned): T={T}  H={H}  W={W}  (float32)')
print(f'Intensity range : [{video_raw.min():.1f}, {video_raw.max():.1f}]  '
      f'mean={video_raw.mean():.1f}')

# ── Step 1: flat-field (÷ temporal median) ──────────────────────────────────
print('\nStep 1: flat-field (/ temporal median) ...')
median_xy = np.median(video_raw, axis=0)           # (H, W)
video_ff  = video_raw / (median_xy[np.newaxis] + 1e-10)
print(f'  After flat-field: mean={video_ff.mean():.4f}  std={video_ff.std():.6f}')

# ── Step 2: per-frame Gaussian background division ──────────────────────────
print('Step 2: Gaussian BG division (sigma=4, per frame) ...')
video_proc = np.empty_like(video_ff)
for t in range(T):
    bg = gaussian_filter(video_ff[t], sigma=4)
    video_proc[t] = video_ff[t] / (bg + 1e-10)
    if t % max(1, T // 5) == 0:
        print(f'  frame {t:4d}/{T}')

print(f'\nProcessed: mean={video_proc.mean():.4f}  std={video_proc.std():.6f}  '
      f'range=[{video_proc.min():.4f}, {video_proc.max():.4f}]')
print('>>> mean ≈ 1.0  →  G(τ) denominator <I>² is valid ✓')

# ── Signal centering check ──────────────────────────────────────────────────
sig_zero = video_proc[0] - 1.0        # zero-centred fluctuation (visualisation only)
abs_99   = np.percentile(np.abs(sig_zero), 99)
print(f'\nFluctuation frame (δI/I): mean={sig_zero.mean():.6f}  '
      f'std={sig_zero.std():.6f}  |p99|={abs_99:.6f}')

# ── 3-panel plot ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1 — raw binned
p1, p99 = np.percentile(video_raw[0], 1), np.percentile(video_raw[0], 99)
im0 = axes[0].imshow(video_raw[0], cmap='gray', vmin=p1, vmax=p99)
plt.colorbar(im0, ax=axes[0], label='Intensity')
axes[0].set_title(f'Frame 0 — raw binned ({H}×{W})')
axes[0].axis('off')

# Panel 2 — after flat-field (mean≈1, used for G computation)
p1, p99 = np.percentile(video_proc[0], 1), np.percentile(video_proc[0], 99)
im1 = axes[1].imshow(video_proc[0], cmap='gray', vmin=p1, vmax=p99)
plt.colorbar(im1, ax=axes[1], label='I/<I>')
axes[1].set_title('Frame 0 — after flat-field + BG  (mean≈1, G input)')
axes[1].axis('off')

# Panel 3 — zero-centred fluctuation (visualisation; NOT fed to G formula)
abs_99v = np.percentile(np.abs(sig_zero), 99)
im2 = axes[2].imshow(sig_zero, cmap='RdBu_r', vmin=-abs_99v, vmax=abs_99v)
plt.colorbar(im2, ax=axes[2], label='δI/I')
axes[2].set_title('Frame 0 — fluctuation δI/I  (video_proc − 1, viz only)')
axes[2].axis('off')

plt.tight_layout()
plt.show()

# ── CV distribution (cell-detection sanity check) ────────────────────────────
mean_I = video_proc.mean(axis=0)
cv_map = video_proc.std(axis=0) / (mean_I + 1e-10)
print(f'\nCV map:  mean={cv_map.mean():.4f}  '
      f'p5={np.percentile(cv_map, 5):.4f}  '
      f'p50={np.percentile(cv_map, 50):.4f}  '
      f'p95={np.percentile(cv_map, 95):.4f}')
print(f'Cell pixels (CV≥0.005): {(cv_map >= 0.005).sum()} / {cv_map.size}  '
      f'({100*(cv_map>=0.005).mean():.1f}%)')
print('\nPreprocessing done.')


In [ ]:
# ── Real Video Inference ───────────────────────────────────────────────────
import sys, os, numpy as np

# Clear HuggingFace 'datasets' from cache so local package is found
for _k in list(sys.modules.keys()):
    if _k == 'datasets' or _k.startswith('datasets.'):
        del sys.modules[_k]

import torch
import matplotlib.pyplot as plt
from datasets.phys_recon_dataset import PhysReconDataset
from models.pissl_tau_encoder    import PISSLTauEncoder

RECON_TAUS = (1, 2, 4, 8, 16, 32, 48, 64, 96, 128)
CKPT_PATH  = f'./checkpoint/pissl_phys_recon_{VERSION}.pth'
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Compute G_empirical (mode=eval → full frame, no blind-spot)
# video_proc has mean≈1.0 — required for G(τ) = <δI·δI(τ)> / <I>²
print('Computing G_empirical ...')
real_ds = PhysReconDataset(
    video_tensor = video_proc,
    recon_taus   = RECON_TAUS,
    patch_size   = 64,
    mode         = 'eval',
)

# Load trained model
print(f'Loading: {CKPT_PATH}')
model = PISSLTauEncoder(recon_taus=RECON_TAUS, predict_amplitude=False).to(device)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval()

# Pad to multiple of 8 (3× MaxPool2d in encoder)
full_input = real_ds[0]                                   # (K, H, W)
K, Hv, Wv  = full_input.shape
pad_h = (8 - Hv % 8) % 8
pad_w = (8 - Wv % 8) % 8
if pad_h or pad_w:
    import torch.nn.functional as F
    full_input = F.pad(full_input, (0, pad_w, 0, pad_h))
    print(f'Padded input: {Hv}×{Wv} → {Hv+pad_h}×{Wv+pad_w}')

with torch.no_grad():
    preds_real = model(full_input.unsqueeze(0).to(device))  # (1, 2, H', W')

gamma_pred = preds_real[0, 0].cpu().numpy()[:Hv, :Wv]
alpha_pred = preds_real[0, 1].cpu().numpy()[:Hv, :Wv]
cell_mask  = real_ds.cell_mask

# ── Plot inference maps (percentile-clipped) ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, data, cmap, title in [
    (axes[0], np.where(cell_mask, gamma_pred, np.nan), 'magma',  f'Model gamma [{VERSION}]'),
    (axes[1], np.where(cell_mask, alpha_pred, np.nan), 'plasma', f'Model alpha [{VERSION}]'),
]:
    p1  = np.nanpercentile(data, 1)
    p99 = np.nanpercentile(data, 99)
    im  = ax.imshow(data, cmap=cmap, vmin=p1, vmax=p99)
    plt.colorbar(im, ax=ax, label=f'[{p1:.3f}, {p99:.3f}]')
    ax.set_title(title)
    ax.axis('off')
plt.suptitle('Model Inference — Real Cell  (1st–99th percentile clip)', fontsize=13)
plt.tight_layout()
plt.show()

# ── G_norm channel plots (diagnostic: are autocorrelation channels clean?) ───
g_norm_np = full_input[:K].numpy()[:, :Hv, :Wv]    # (K, H, W) — masked G_norm
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for i, (ax, tau) in enumerate(zip(axes.flat, RECON_TAUS)):
    ch = np.where(cell_mask, g_norm_np[i], np.nan)
    p1, p99 = np.nanpercentile(ch, 1), np.nanpercentile(ch, 99)
    im = ax.imshow(ch, cmap='inferno', vmin=p1, vmax=p99)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(f'G_norm  τ={tau}', fontsize=9)
    ax.axis('off')
plt.suptitle('G_norm channels fed to model', fontsize=12)
plt.tight_layout()
plt.show()

# ── Statistics ───────────────────────────────────────────────────────────────
g_cell = gamma_pred[cell_mask]
a_cell = alpha_pred[cell_mask]
print(f'\nCell pixels : {cell_mask.sum()}  ({100*cell_mask.mean():.1f}% of frame)')
print(f'Model gamma : mean={g_cell.mean():.3f}  std={g_cell.std():.3f}  '
      f'p1={np.percentile(g_cell,1):.3f}  p99={np.percentile(g_cell,99):.3f}')
print(f'Model alpha : mean={a_cell.mean():.3f}  std={a_cell.std():.3f}  '
      f'p1={np.percentile(a_cell,1):.3f}  p99={np.percentile(a_cell,99):.3f}')
print('Inference done.')


In [ ]:
# ── Compare Model vs Traditional iSCORS + Save Results ─────────────────────
import numpy as np
import matplotlib.pyplot as plt
import tifffile, os
from skimage.transform import resize as sk_resize

cell_mask = real_ds.cell_mask

def _match(arr, shape):
    """Resize reference map to match model output shape if needed."""
    return arr if arr.shape == shape else sk_resize(
        arr, shape, preserve_range=True, anti_aliasing=True).astype(np.float32)

def _pclip(data, lo=1, hi=99):
    """Return (p_lo, p_hi) using nanpercentile; robust to NaN cell mask."""
    return np.nanpercentile(data, lo), np.nanpercentile(data, hi)

# ── Build panel list ─────────────────────────────────────────────────────────
gm_data = np.where(cell_mask, gamma_pred, np.nan)
am_data = np.where(cell_mask, alpha_pred, np.nan)
gp1, gp99 = _pclip(gm_data)
ap1, ap99 = _pclip(am_data)

panels = [
    (f'Model gamma [{VERSION}]', gm_data, 'magma',  gp1, gp99),
    (f'Model alpha [{VERSION}]', am_data, 'plasma', ap1, ap99),
]

gr_matched = ar_matched = None
if gamma_ref is not None:
    gr_matched = _match(gamma_ref, gamma_pred.shape)
    rp1, rp99  = _pclip(gr_matched)
    panels.append(('iSCORS gamma (ref)', gr_matched, 'magma',  rp1, rp99))
if alpha_ref is not None:
    ar_matched = _match(alpha_ref, alpha_pred.shape)
    rp1, rp99  = _pclip(ar_matched)
    panels.append(('iSCORS alpha (ref)', ar_matched, 'plasma', rp1, rp99))

# ── Side-by-side map comparison ──────────────────────────────────────────────
fig_cmp, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5))
if len(panels) == 1:
    axes = [axes]
for ax, (title, data, cmap, vmin, vmax) in zip(axes, panels):
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
                 label=f'[{vmin:.3f}, {vmax:.3f}]')
    ax.set_title(title, fontsize=10)
    ax.axis('off')
fig_cmp.suptitle('Model vs Traditional iSCORS  (1st–99th percentile clip)', fontsize=13)
plt.tight_layout()
plt.show()

# ── Scatter: model vs iSCORS (on cell pixels) ────────────────────────────────
if gr_matched is not None and ar_matched is not None:
    gr_m = gr_matched[cell_mask]
    ar_m = ar_matched[cell_mask]
    g_m  = gamma_pred[cell_mask]
    a_m  = alpha_pred[cell_mask]

    fig_sc, axes_sc = plt.subplots(1, 2, figsize=(10, 4))
    for ax, x, y, param, cmp in [
        (axes_sc[0], gr_m, g_m, 'gamma', 'magma'),
        (axes_sc[1], ar_m, a_m, 'alpha', 'plasma'),
    ]:
        ax.scatter(x, y, s=0.5, alpha=0.3, c=y, cmap=cmp)
        lim = [min(x.min(), y.min()), max(x.max(), y.max())]
        ax.plot(lim, lim, 'r--', lw=1.2, label='y=x')
        ax.set_xlabel(f'iSCORS {param}')
        ax.set_ylabel(f'Model {param}')
        ax.set_title(f'{param}: model vs iSCORS')
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

# ── Numerical comparison ─────────────────────────────────────────────────────
stats_lines = [f'=== Real Cell Analysis [{VERSION}] ===',
               f'Video      : {VIDEO_PATH}',
               f'Frames used: {T}   Cell pixels: {cell_mask.sum()}  '
               f'({100*cell_mask.mean():.1f}%)',
               '',
               '--- Model predictions ---',
               f'  gamma : mean={g_cell.mean():.4f}  std={g_cell.std():.4f}  '
               f'p1={np.percentile(g_cell,1):.4f}  p99={np.percentile(g_cell,99):.4f}',
               f'  alpha : mean={a_cell.mean():.4f}  std={a_cell.std():.4f}  '
               f'p1={np.percentile(a_cell,1):.4f}  p99={np.percentile(a_cell,99):.4f}']

if gr_matched is not None and ar_matched is not None:
    from scipy.stats import pearsonr
    gr_m = gr_matched[cell_mask]
    ar_m = ar_matched[cell_mask]
    mae_g = np.abs(gamma_pred[cell_mask] - gr_m).mean()
    mae_a = np.abs(alpha_pred[cell_mask] - ar_m).mean()
    r_g, _  = pearsonr(gamma_pred[cell_mask], gr_m)
    r_a, _  = pearsonr(alpha_pred[cell_mask], ar_m)
    stats_lines += ['',
        '--- vs Traditional iSCORS ---',
        f'  gamma  MAE={mae_g:.4f}   Pearson r={r_g:.4f}',
        f'  alpha  MAE={mae_a:.4f}   Pearson r={r_a:.4f}']

print('\n'.join(stats_lines))

# ── Save everything to Drive ─────────────────────────────────────────────────
os.makedirs(SAVE_DIR, exist_ok=True)

tifffile.imwrite(os.path.join(SAVE_DIR, f'model_gamma_{VERSION}.tif'),
                 np.where(cell_mask, gamma_pred, 0).astype(np.float32))
tifffile.imwrite(os.path.join(SAVE_DIR, f'model_alpha_{VERSION}.tif'),
                 np.where(cell_mask, alpha_pred, 0).astype(np.float32))

_stats_path = os.path.join(SAVE_DIR, f'real_stats_{VERSION}.txt')
with open(_stats_path, 'w') as f:
    f.write('\n'.join(stats_lines) + '\n')

_cmp_path = os.path.join(SAVE_DIR, f'real_comparison_{VERSION}.png')
fig_cmp.savefig(_cmp_path, dpi=120, bbox_inches='tight')

print(f'\nSaved → {SAVE_DIR}/')
print(f'  model_gamma_{VERSION}.tif')
print(f'  model_alpha_{VERSION}.tif')
print(f'  real_stats_{VERSION}.txt')
print(f'  real_comparison_{VERSION}.png')


In [ ]:
# ── Download Real Inference Results as ZIP ──────────────────────────────────
import zipfile, os, glob
from google.colab import files

zip_name = f'inference_real_{VERSION}.zip'

# Collect: all PNGs from result/ that belong to this session,
# plus TIFs and stats saved to Drive SAVE_DIR
collected = []

# Local result/ PNGs (synthetic training outputs already in first zip)
# Real-analysis outputs saved to Drive
for pattern in [
    os.path.join(SAVE_DIR, f'*{VERSION}*'),
    os.path.join(SAVE_DIR, 'iscors_gamma_ref.tif'),
    os.path.join(SAVE_DIR, 'iscors_alpha_ref.tif'),
]:
    collected.extend(glob.glob(pattern))

collected = sorted(set(collected))
print(f'Files to zip ({len(collected)}):')
for p in collected:
    print(f'  {os.path.basename(p)}  ({os.path.getsize(p)/1024:.1f} KB)')

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in collected:
        zf.write(p, os.path.basename(p))

print(f'\nCreated {zip_name}  ({os.path.getsize(zip_name)/1024:.1f} KB)')
files.download(zip_name)
